# Chapter 5 — Numerical algorithms (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch05_numerical_algorithms_java.ipynb)

Executable notebook of the chapter “Numerical algorithms”: it contains the algorithms `ChangeOfBase`, `BasicPrimalityTest`, `PrimalityTest`, `SieveOfEratosthenes` and `EuclideanAlgorithm` written in Java and run on the same examples as the lecture notes, printing the results and, where the notes show a trace, the step-by-step trace in the same format.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/algorithms/basic-algorithms/05-numerical-algorithms/). The same notebook is also available in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch05_numerical_algorithms_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch05_numerical_algorithms_cpp.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`. The first cell checks that the JDK is present and installs it if it is missing. Each Java program is written to a file by the `%%writefile Name.java` cell, compiled with `javac` and run with `java` in the next cell. The programs read their input from the command line: the “Try it yourself” cell runs the same program with numbers you can change.

**Indices.** In the pseudo-code arrays start from 1, here from 0: the element $a[i]$ of the pseudo-code is `a[i-1]` in the code. The printed traces use the indices of the code (as in the C++ examples of the lecture notes), so they are those of the pseudo-code with every index decreased by 1. The method names are those of the pseudo-code, in the Java convention (`camelCase`: `findMaximum` for `FindMaximum`).

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!java -version || apt-get -qq install -y default-jdk-headless

openjdk version "25.0.4.1" 2026-08-18
OpenJDK Runtime Environment (build 25.0.4.1+1-1-26.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.4.1+1-1-26.04.4-Ubuntu, mixed mode, sharing)


## ChangeOfBase

**Input:** a natural number $n \ge 1$ (in base 10) and a base $b \ge 2$. **Output:** the array $d[1..nd]$ of the digits of $n$ in base $b$ and the number of digits $nd$.

One division by $b$ extracts the least significant digit ($r = n \bmod b$) and leaves as quotient the number written with the other digits: the division is repeated until the quotient is 0. The digits come out from the least significant to the most significant, so in the array `d` the digit of weight $b^0$ is in `d[0]` (in $d[1]$ in the pseudo-code).

The number of digits $nd = \lfloor \log_b n \rfloor + 1$ is computed here with integer multiplications only, without floating point logarithms: in double precision `log(1000)/log(10)` evaluates to $2.9999999999999996$ and the formula would give 3 digits instead of 4.

In [2]:
%%writefile ChangeOfBase.java
import java.util.Arrays;

public class ChangeOfBase {

    // Returns the digits of n in base b: d[0] is the least significant digit, d.length is nd
    static int[] changeOfBase(int n, int b, boolean trace) {
        int nd = 1;                     // nd = ⌊log_b n⌋ + 1, with integer operations only
        for (long p = b; p <= n; p *= b)
            nd = nd + 1;
        int[] d = new int[nd];          // array of the digits
        int k = 0;                      // k = 1 in the pseudo-code
        while (n > 0) {
            int q = n / b;              // quotient
            int r = n % b;              // remainder (digit in base b)
            d[k] = r;
            n = q;
            k = k + 1;
            if (trace)
                System.out.printf("%10d %5d %3d %5d %3d  %s\n", k, q, r, n, k, Arrays.toString(Arrays.copyOf(d, k)));
        }
        return d;
    }

    static String digitString(int[] d) {    // the digits from the most to the least significant
        StringBuilder s = new StringBuilder();
        for (int i = d.length - 1; i >= 0; i--) s.append("0123456789ABCDEF".charAt(d[i]));
        return s.toString();
    }

    static void run(int n, int b) {
        System.out.printf("Conversion: n = %d, b = %d\n", n, b);
        System.out.printf("%10s %5s %3s %5s %3s  %s\n", "iteration", "q", "r", "n", "nd", "digits");
        int[] d = changeOfBase(n, b, true);
        System.out.printf("Return: nd = %d, d = %s\n", d.length, Arrays.toString(d));
        System.out.printf("%d = (%s)_%d\n\n", n, digitString(d), b);
    }

    public static void main(String[] args) {
        if (args.length > 1) {          // “Your turn”: n and the base b from the command line
            run(Integer.parseInt(args[0]), Integer.parseInt(args[1]));
            return;
        }
        run(7, 2);
        run(8, 2);
        run(25, 3);
        run(100, 8);
        run(1000, 10);    // 4 digits: the case where the floating point log(1000)/log(10) fails
    }
}


Writing ChangeOfBase.java


### Execution on the examples of the chapter

In [3]:
!javac ChangeOfBase.java && java ChangeOfBase

Conversion: n = 7, b = 2
 iteration     q   r     n  nd  digits
         1     3   1     3   1  [1]
         2     1   1     1   2  [1, 1]
         3     0   1     0   3  [1, 1, 1]
Return: nd = 3, d = [1, 1, 1]
7 = (111)_2

Conversion: n = 8, b = 2
 iteration     q   r     n  nd  digits
         1     4   0     4   1  [0]
         2     2   0     2   2  [0, 0]
         3     1   0     1   3  [0, 0, 0]
         4     0   1     0   4  [0, 0, 0, 1]
Return: nd = 4, d = [0, 0, 0, 1]
8 = (1000)_2

Conversion: n = 25, b = 3
 iteration     q   r     n  nd  digits
         1     8   1     8   1  [1]
         2     2   2     2   2  [1, 2]
         3     0   2     0   3  [1, 2, 2]
Return: nd = 3, d = [1, 2, 2]
25 = (221)_3

Conversion: n = 100, b = 8
 iteration     q   r     n  nd  digits
         1    12   4    12   1  [4]
         2     1   4     1   2  [4, 4]
         3     0   1     0   3  [4, 4, 1]
Return: nd = 3, d = [4, 4, 1]
100 = (144)_8

Conversion: n = 1000, b = 10
 iteration     q   r

### Try it yourself

Change the numbers below and run the cell again. The numbers are $n$ and the base $b$ (up to 16: the digits above 9 are written A, B, …).

In [4]:
!java ChangeOfBase 255 16

Conversion: n = 255, b = 16
 iteration     q   r     n  nd  digits
         1    15  15    15   1  [15]
         2     0  15     0   2  [15, 15]
Return: nd = 2, d = [15, 15]
255 = (FF)_16



## BasicPrimalityTest

**Input:** a natural number $n$. **Output:** `true` if $n$ is prime, `false` otherwise.

A divisor $d < n$ of $n$ is at most $n/2$: it suffices to test the candidates $d = 2, 3, \dots, \lfloor n/2 \rfloor$. For a prime number the loop is executed completely: for $n = 29$ the candidates are 13. The function also returns the number of candidates tested.

In [5]:
%%writefile BasicPrimalityTest.java
public class BasicPrimalityTest {

    static int tested;                  // candidates tested by the last call

    // Returns true if n is prime
    static boolean basicPrimalityTest(int n, boolean trace) {
        tested = 0;
        if (n < 2) {
            if (trace)
                System.out.printf("%3s %8s  %s\n", "-", "-", "n < 2: return false");
            return false;               // 0 and 1 are not prime
        }
        for (int d = 2; d <= n / 2; d++) {  // d = 2..⌊n/2⌋
            tested++;
            if (n % d == 0) {
                if (trace)
                    System.out.printf("%3d %8d  %s\n", d, n % d, "return false: d divides n");
                return false;           // d divides n: not prime
            }
            if (trace)
                System.out.printf("%3d %8d  %s\n", d, n % d, "continue");
        }
        if (trace)
            System.out.printf("%3s %8s  %s\n", "-", "-", "the loop ends: return true");
        return true;                    // no divisor found: n is prime
    }

    static void run(int n) {
        System.out.printf("Check: n = %d, floor(n/2) = %d\n", n, n / 2);
        System.out.printf("%3s %8s  %s\n", "d", "n mod d", "action");
        boolean result = basicPrimalityTest(n, true);
        System.out.printf("Return: %s, candidates tested: %d\n\n", result ? "true" : "false", tested);
    }

    public static void main(String[] args) {
        if (args.length > 0) {          // “Your turn”: the numbers to check come from the command line
            for (String s : args) run(Integer.parseInt(s));
            return;
        }
        run(36);
        run(29);
        run(25);
    }
}


Writing BasicPrimalityTest.java


### Execution on the examples of the chapter

In [6]:
!javac BasicPrimalityTest.java && java BasicPrimalityTest

Check: n = 36, floor(n/2) = 18
  d  n mod d  action
  2        0  return false: d divides n
Return: false, candidates tested: 1

Check: n = 29, floor(n/2) = 14
  d  n mod d  action
  2        1  continue
  3        2  continue
  4        1  continue
  5        4  continue
  6        5  continue
  7        1  continue
  8        5  continue
  9        2  continue
 10        9  continue
 11        7  continue
 12        5  continue
 13        3  continue
 14        1  continue
  -        -  the loop ends: return true
Return: true, candidates tested: 13

Check: n = 25, floor(n/2) = 12
  d  n mod d  action
  2        1  continue
  3        1  continue
  4        1  continue
  5        0  return false: d divides n
Return: false, candidates tested: 4



### Try it yourself

Change the numbers below and run the cell again. The number to check is $n$.

In [7]:
!java BasicPrimalityTest 97

Check: n = 97, floor(n/2) = 48
  d  n mod d  action
  2        1  continue
  3        1  continue
  4        1  continue
  5        2  continue
  6        1  continue
  7        6  continue
  8        1  continue
  9        7  continue
 10        7  continue
 11        9  continue
 12        1  continue
 13        6  continue
 14       13  continue
 15        7  continue
 16        1  continue
 17       12  continue
 18        7  continue
 19        2  continue
 20       17  continue
 21       13  continue
 22        9  continue
 23        5  continue
 24        1  continue
 25       22  continue
 26       19  continue
 27       16  continue
 28       13  continue
 29       10  continue
 30        7  continue
 31        4  continue
 32        1  continue
 33       31  continue
 34       29  continue
 35       27  continue
 36       25  continue
 37       23  continue
 38       21  continue
 39       19  continue
 40       17  continue
 41       15  continue
 42       13  continue
 43  

## PrimalityTest

**Input:** a natural number $n$. **Output:** `true` if $n$ is prime, `false` otherwise.

The divisors come in pairs $d$ and $n/d$ and in each pair one of the two numbers is at most $\sqrt{n}$: a number $n \ge 2$ is prime if and only if no number in $\{2, 3, \dots, \lfloor \sqrt{n} \rfloor\}$ divides it. For $n = 2$ and $n = 3$ the loop is not executed and the function returns `true`. For $n = 29$ the candidates are 4, instead of the 13 of the basic algorithm.

In [8]:
%%writefile PrimalityTest.java
public class PrimalityTest {

    static int tested;                  // candidates tested by the last call

    // Returns true if n is prime
    static boolean primalityTest(int n, boolean trace) {
        tested = 0;
        if (n < 2) {
            if (trace)
                System.out.printf("%3s %8s  %s\n", "-", "-", "n < 2: return false");
            return false;               // 0 and 1 are not prime
        }
        for (int d = 2; d <= (int) Math.sqrt(n); d++) {     // d = 2..⌊√n⌋
            tested++;
            if (n % d == 0) {
                if (trace)
                    System.out.printf("%3d %8d  %s\n", d, n % d, "return false: d divides n");
                return false;           // d divides n: not prime
            }
            if (trace)
                System.out.printf("%3d %8d  %s\n", d, n % d, "continue");
        }
        if (trace)
            System.out.printf("%3s %8s  %s\n", "-", "-", "the loop ends: return true");
        return true;                    // no divisor found: n is prime
    }

    static void run(int n) {
        System.out.printf("Check: n = %d, floor(sqrt(n)) = %d\n", n, (int) Math.sqrt(n));
        System.out.printf("%3s %8s  %s\n", "d", "n mod d", "action");
        boolean result = primalityTest(n, true);
        System.out.printf("Return: %s, candidates tested: %d\n\n", result ? "true" : "false", tested);
    }

    public static void main(String[] args) {
        if (args.length > 0) {          // “Your turn”: the numbers to check come from the command line
            for (String s : args) run(Integer.parseInt(s));
            return;
        }
        run(36);
        run(29);
        run(25);
        run(2);         // the loop is not executed
    }
}


Writing PrimalityTest.java


### Execution on the examples of the chapter

In [9]:
!javac PrimalityTest.java && java PrimalityTest

Check: n = 36, floor(sqrt(n)) = 6
  d  n mod d  action
  2        0  return false: d divides n
Return: false, candidates tested: 1

Check: n = 29, floor(sqrt(n)) = 5
  d  n mod d  action
  2        1  continue
  3        2  continue
  4        1  continue
  5        4  continue
  -        -  the loop ends: return true
Return: true, candidates tested: 4

Check: n = 25, floor(sqrt(n)) = 5
  d  n mod d  action
  2        1  continue
  3        1  continue
  4        1  continue
  5        0  return false: d divides n
Return: false, candidates tested: 4

Check: n = 2, floor(sqrt(n)) = 1
  d  n mod d  action
  -        -  the loop ends: return true
Return: true, candidates tested: 0



### Try it yourself

Change the numbers below and run the cell again. The number to check is $n$.

In [10]:
!java PrimalityTest 97

Check: n = 97, floor(sqrt(n)) = 9
  d  n mod d  action
  2        1  continue
  3        1  continue
  4        1  continue
  5        2  continue
  6        1  continue
  7        6  continue
  8        1  continue
  9        7  continue
  -        -  the loop ends: return true
Return: true, candidates tested: 8



## SieveOfEratosthenes

**Input:** a natural number $n \ge 2$. **Output:** a boolean array $prime[1..n]$ where $prime[i] = 1$ if and only if $i$ is prime.

We start with all the numbers marked as prime (except 1) and, for every prime $p \le \lfloor \sqrt{n} \rfloor$, we mark its multiples as composite starting from $p^2$. In Python, C++ and Java the array starts from 0: `prime[i]` is associated with the number $i$, so it has $n+1$ entries `prime[0..n]` and `prime[0]` (the number 0, not prime) is 0.

In [11]:
%%writefile SieveOfEratosthenes.java
import java.util.ArrayList;
import java.util.Arrays;
import java.util.List;

public class SieveOfEratosthenes {

    // Returns prime[0..n]: prime[i] = 1 if and only if i is prime (n >= 2)
    static int[] sieveOfEratosthenes(int n, boolean trace) {
        int[] prime = new int[n + 1];
        Arrays.fill(prime, 1);          // prime[0..n], all set to 1
        prime[0] = 0;                   // 0 is not prime
        prime[1] = 0;                   // 1 is not prime
        if (trace)
            System.out.printf("%5s  %s\n", "start", join(prime));

        for (int p = 2; p <= (int) Math.sqrt(n); p++) {     // p = 2..⌊√n⌋
            List<Integer> js = new ArrayList<>();           // the values of j (only for the trace)
            if (prime[p] == 1) {
                // mark the multiples of p starting from p^2
                for (int j = p * p; j <= n; j += p) {
                    prime[j] = 0;       // j is not prime
                    js.add(j);
                }
            }
            if (trace)
                System.out.printf("%5d  %s  %s\n", p, join(prime),
                        js.isEmpty() ? "no inner loop: prime[" + p + "] is 0"
                                     : "j = " + js.toString().replace("[", "").replace("]", ""));
        }
        return prime;
    }

    static String join(int[] a) {       // the entries separated by spaces (only for the trace)
        StringBuilder s = new StringBuilder();
        for (int i = 0; i < a.length; i++) s.append(i > 0 ? " " : "").append(a[i]);
        return s.toString();
    }

    static void run(int n) {
        System.out.printf("Sieve: n = %d, floor(sqrt(n)) = %d\n", n, (int) Math.sqrt(n));
        System.out.printf("%5s  prime[0..%d]\n", "p", n);
        int[] prime = sieveOfEratosthenes(n, true);
        List<Integer> primes = new ArrayList<>();
        for (int i = 0; i <= n; i++)
            if (prime[i] == 1) primes.add(i);
        System.out.printf("Prime numbers <= %d: %s\n\n", n, primes);
    }

    public static void main(String[] args) {
        if (args.length > 0) {          // “Your turn”: n from the command line
            run(Integer.parseInt(args[0]));
            return;
        }
        run(10);
        run(20);
    }
}


Writing SieveOfEratosthenes.java


### Execution on the examples of the chapter

In [12]:
!javac SieveOfEratosthenes.java && java SieveOfEratosthenes

Sieve: n = 10, floor(sqrt(n)) = 3
    p  prime[0..10]
start  0 0 1 1 1 1 1 1 1 1 1
    2  0 0 1 1 0 1 0 1 0 1 0  j = 4, 6, 8, 10
    3  0 0 1 1 0 1 0 1 0 0 0  j = 9
Prime numbers <= 10: [2, 3, 5, 7]

Sieve: n = 20, floor(sqrt(n)) = 4
    p  prime[0..20]
start  0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
    2  0 0 1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0  j = 4, 6, 8, 10, 12, 14, 16, 18, 20
    3  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0  j = 9, 12, 15, 18
    4  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0  no inner loop: prime[4] is 0
Prime numbers <= 20: [2, 3, 5, 7, 11, 13, 17, 19]



### Try it yourself

Change the numbers below and run the cell again. The number is $n$ (at least 2).

In [13]:
!java SieveOfEratosthenes 30

Sieve: n = 30, floor(sqrt(n)) = 5
    p  prime[0..30]
start  0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
    2  0 0 1 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0  j = 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30
    3  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0  j = 9, 12, 15, 18, 21, 24, 27, 30
    4  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0  no inner loop: prime[4] is 0
    5  0 0 1 1 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 1 0 0 0 1 0 0 0 0 0 1 0  j = 25, 30
Prime numbers <= 30: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]



## EuclideanAlgorithm

**Input:** two natural numbers $n, m \ge 1$. **Output:** $\gcd(n, m)$, the greatest common divisor.

The pair $(n, m)$ is replaced with $(m,\ n \bmod m)$, which has the same common divisors, until the second number is 0: then $\gcd(n, 0) = n$. The parameters are passed by value: the function modifies its own copies. Two numbers are coprime if and only if the result is 1.

In [14]:
%%writefile EuclideanAlgorithm.java
public class EuclideanAlgorithm {

    // Returns the greatest common divisor of n and m (n, m >= 1)
    static int euclideanAlgorithm(int n, int m, boolean trace) {
        int step = 0;                   // only for the trace
        while (m != 0) {
            step++;
            int r = n % m;
            if (trace)
                System.out.printf("%5d %5d %5d  %d mod %d = %d\n", step, n, m, n, m, r);
            n = m;
            m = r;
        }
        if (trace)
            System.out.printf("%5d %5d %5d  %s\n", step + 1, n, m, "--- (the loop ends)");
        return n;                       // m = 0: n is the GCD
    }

    static void run(int n, int m) {
        System.out.printf("GCD computation: n = %d, m = %d\n", n, m);
        System.out.printf("%5s %5s %5s  %s\n", "step", "n", "m", "r = n mod m");
        int g = euclideanAlgorithm(n, m, true);     // n and m are unchanged: passed by value
        System.out.printf("Return: gcd(%d, %d) = %d\n", n, m, g);
        System.out.printf("coprime: %s\n\n", g == 1 ? "true" : "false");
    }

    public static void main(String[] args) {
        if (args.length > 1) {          // “Your turn”: the two numbers from the command line
            run(Integer.parseInt(args[0]), Integer.parseInt(args[1]));
            return;
        }
        run(48, 18);
        run(252, 105);
        run(35, 12);
    }
}


Writing EuclideanAlgorithm.java


### Execution on the examples of the chapter

In [15]:
!javac EuclideanAlgorithm.java && java EuclideanAlgorithm

GCD computation: n = 48, m = 18
 step     n     m  r = n mod m
    1    48    18  48 mod 18 = 12
    2    18    12  18 mod 12 = 6
    3    12     6  12 mod 6 = 0
    4     6     0  --- (the loop ends)
Return: gcd(48, 18) = 6
coprime: false

GCD computation: n = 252, m = 105
 step     n     m  r = n mod m
    1   252   105  252 mod 105 = 42
    2   105    42  105 mod 42 = 21
    3    42    21  42 mod 21 = 0
    4    21     0  --- (the loop ends)
Return: gcd(252, 105) = 21
coprime: false

GCD computation: n = 35, m = 12
 step     n     m  r = n mod m
    1    35    12  35 mod 12 = 11
    2    12    11  12 mod 11 = 1
    3    11     1  11 mod 1 = 0
    4     1     0  --- (the loop ends)
Return: gcd(35, 12) = 1
coprime: true



### Try it yourself

Change the numbers below and run the cell again. The two numbers are $n$ and $m$.

In [16]:
!java EuclideanAlgorithm 1071 462

GCD computation: n = 1071, m = 462
 step     n     m  r = n mod m
    1  1071   462  1071 mod 462 = 147
    2   462   147  462 mod 147 = 21
    3   147    21  147 mod 21 = 0
    4    21     0  --- (the loop ends)
Return: gcd(1071, 462) = 21
coprime: false



---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).